<a href="https://colab.research.google.com/github/hinojosaM/machine_learning_homework3/blob/main/machine_learning_homework3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# Descargamos el archivo CSV directamente del repositorio oficial con wget
!wget -q -N https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

import pandas as pd
import numpy as np

# Cargamos el dataset en un DataFrame
df = pd.read_csv('course_lead_scoring_2026.csv')

# Inspeccionamos las primeras filas y dimensiones
print(f"Dimensiones del dataset: {df.shape}")
df.head()

Dimensiones del dataset: (5000, 9)


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [20]:
target = 'converted'
features = [col for col in df.columns if col != target]

categorical = list(df[features].select_dtypes(include=['object']).columns)
numerical = list(df[features].select_dtypes(include=['number']).columns)

print("Variables categóricas:", categorical)
print("Variables numéricas:", numerical)
print("\nValores nulos antes de imputar:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Reemplazamos nulos según las instrucciones:
# - Categóricos -> 'NA'
# - Numéricos -> 0.0
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

# Verificamos que no queden valores faltantes
assert df.isnull().sum().sum() == 0, "Aún quedan valores nulos"
print("\nImputación completada con éxito. Nulos restantes: 0")

Variables categóricas: ['lead_source', 'industry', 'employment_status', 'location']
Variables numéricas: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

Valores nulos antes de imputar:
lead_source          148
industry             240
employment_status    194
location             208
annual_income        369
lead_score            35
dtype: int64

Imputación completada con éxito. Nulos restantes: 0


In [21]:
# Calculamos la frecuencia de cada valor en la columna
print("Distribución de valores en 'industry':")
print(df['industry'].value_counts())

mode_industry = df['industry'].mode()[0]
print(f"\n-->  Q1: La observación más frecuente (moda) es: '{mode_industry}'")

Distribución de valores en 'industry':
industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

-->  Q1: La observación más frecuente (moda) es: 'technology'


In [22]:
# Calculamos la matriz de correlación de Pearson
corr_matrix = df[numerical].corr()
print("Matriz de correlación:")
display(corr_matrix)

# Evaluamos específicamente los 4 pares dados en las opciones
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

print("\nCorrelación entre los pares candidatos:")
best_pair = None
max_corr = -1

for feat1, feat2 in pairs:
    # Usamos valor absoluto o valor directo según corresponda
    c_val = corr_matrix.loc[feat1, feat2]
    print(f" - {feat1} & {feat2}: {c_val:.4f}")
    if abs(c_val) > max_corr:
        max_corr = abs(c_val)
        best_pair = f"{feat1} and {feat2}"

print(f"\n-->  Q2: El par con mayor correlación es: '{best_pair}'")

Matriz de correlación:


,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000



Correlación entre los pares candidatos:
 - interaction_count & lead_score: 0.9157
 - number_of_courses_viewed & lead_score: 0.7572
 - number_of_courses_viewed & interaction_count: 0.7216
 - annual_income & interaction_count: 0.1228

-->  Q2: El par con mayor correlación es: 'interaction_count and lead_score'


In [23]:
from sklearn.model_selection import train_test_split

# 1. Separamos 80% train+val y 20% test
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

# 2. Del 80%, tomamos 25% para validación (lo que equivale al 20% del total) y 75% train (60% total)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

# Reiniciamos índices para evitar desajustes
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# Extraemos la variable objetivo 'converted' y la eliminamos de los dataframes
y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

del df_train[target]
del df_val[target]
del df_test[target]

print(f"Registros en train: {len(df_train)}")
print(f"Registros en val:   {len(df_val)}")
print(f"Registros en test:  {len(df_test)}")

Registros en train: 3000
Registros en val:   1000
Registros en test:  1000


In [24]:
from sklearn.metrics import mutual_info_score

# Función auxiliar para calcular y redondear el puntaje
def calculate_mi(series):
    return mutual_info_score(series, y_train)

# Calculamos el puntaje únicamente con el conjunto de entrenamiento
mi_scores = df_train[categorical].apply(calculate_mi)
mi_scores_rounded = mi_scores.round(2).sort_values(ascending=False)

print("Puntajes de Información Mutua (sin redondear y redondeados a 2 decimales):")
for col in mi_scores.index:
    print(f" - {col:18s}: {mi_scores[col]:.6f} (redondeado: {round(mi_scores[col], 2)})")

best_mi_feature = mi_scores.idxmax()
print(f"\n-->  Q3: La variable con mayor score de información mutua es: '{best_mi_feature}'")

Puntajes de Información Mutua (sin redondear y redondeados a 2 decimales):
 - lead_source       : 0.034346 (redondeado: 0.03)
 - industry          : 0.001995 (redondeado: 0.0)
 - employment_status : 0.018966 (redondeado: 0.02)
 - location          : 0.001043 (redondeado: 0.0)

-->  Q3: La variable con mayor score de información mutua es: 'lead_source'


In [25]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Convertimos los DataFrames a diccionarios para el DictVectorizer
train_dicts = df_train[categorical + numerical].to_dict(orient='records')
val_dicts = df_val[categorical + numerical].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

# Entrenamos el modelo con los parámetros requeridos
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Predicción en el conjunto de validación
y_pred = model.predict(X_val)

base_accuracy = accuracy_score(y_val, y_pred)
print(f"Accuracy exacto en validación: {base_accuracy:.4f}")
print(f"Accuracy redondeado a 2 decimales: {round(base_accuracy, 2)}")
print(f"\n-->  Q4: {round(base_accuracy, 2)}")

Accuracy exacto en validación: 0.6450
Accuracy redondeado a 2 decimales: 0.65

-->  Q4: 0.65


In [26]:
all_features = categorical + numerical
diffs = {}

# Evaluamos el impacto de excluir cada característica una por una
for feature in all_features:
    subset = [f for f in all_features if f != feature]

    # Vectorización sin la característica seleccionada
    train_dict_sub = df_train[subset].to_dict(orient='records')
    val_dict_sub = df_val[subset].to_dict(orient='records')

    dv_sub = DictVectorizer(sparse=False)
    X_tr_sub = dv_sub.fit_transform(train_dict_sub)
    X_v_sub = dv_sub.transform(val_dict_sub)

    # Entrenamiento con los mismos hiperparámetros
    m_sub = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    m_sub.fit(X_tr_sub, y_train)

    acc_sub = accuracy_score(y_val, m_sub.predict(X_v_sub))
    diff = base_accuracy - acc_sub
    diffs[feature] = diff

print("Diferencias calculadas (base_accuracy - accuracy_sin_feature):")
for f, d in diffs.items():
    print(f" - Sin '{f}': diff = {d:+.5f}")

# Evaluamos las 3 opciones dadas en la pregunta
q5_candidates = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
print("\nComparación entre las opciones de la pregunta:")
for cand in q5_candidates:
    print(f" - '{cand}': diff = {diffs[cand]:+.5f}")

smallest_feature = min(q5_candidates, key=lambda x: diffs[x])
print(f"\n-->  Q5: La característica con la menor diferencia es: '{smallest_feature}'")

Diferencias calculadas (base_accuracy - accuracy_sin_feature):
 - Sin 'lead_source': diff = +0.00300
 - Sin 'industry': diff = +0.00000
 - Sin 'employment_status': diff = +0.00000
 - Sin 'location': diff = +0.00000
 - Sin 'annual_income': diff = -0.07900
 - Sin 'number_of_courses_viewed': diff = +0.00200
 - Sin 'interaction_count': diff = +0.04400
 - Sin 'lead_score': diff = +0.00100

Comparación entre las opciones de la pregunta:
 - 'lead_source': diff = +0.00300
 - 'number_of_courses_viewed': diff = +0.00200
 - 'interaction_count': diff = +0.04400

-->  Q5: La característica con la menor diferencia es: 'number_of_courses_viewed'


In [27]:
c_values = [0.000001, 0.00001, 0.0001, 0.001]
c_results = {}

for c in c_values:
    m_c = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    m_c.fit(X_train, y_train)
    acc_c = accuracy_score(y_val, m_c.predict(X_val))
    acc_rounded = round(acc_c, 3)
    c_results[c] = (acc_c, acc_rounded)
    print(f"C = {c:<10}: Accuracy = {acc_c:.5f} (Redondeado a 3 dec: {acc_rounded:.3f})")

# Determinamos el mejor C (si hay empates, elegimos el menor C)
best_score = max(r[1] for r in c_results.values())
best_c_list = [c for c, r in c_results.items() if r[1] == best_score]
selected_c = min(best_c_list)

print(f"\n-->  Q6: El mejor valor de es: {selected_c}")

C = 1e-06     : Accuracy = 0.59800 (Redondeado a 3 dec: 0.598)
C = 1e-05     : Accuracy = 0.59800 (Redondeado a 3 dec: 0.598)
C = 0.0001    : Accuracy = 0.61300 (Redondeado a 3 dec: 0.613)
C = 0.001     : Accuracy = 0.64500 (Redondeado a 3 dec: 0.645)

-->  Q6: El mejor valor de es: 0.001
